# Sunrise Social Club: Predictive Modeling

**Objective:** Predict beverage demand to support inventory planning and event preparation.

### Loading the Data and Overview

Out of the gate, our biggest operational challenge was determining how much cold brew, matcha, and lemonade to batch for each market. Sometimes we prepared too little and sold out early, while other times we overproduced and had product go to waste. This predictive model estimates demand for each beverage at a given event and converts those predictions into recommended batch sizes in gallons. Because fresh beverages can be prepared throughout the day, the forecasts are intended to serve as a starting point for inventory planning. The goal is to batch conservatively before the event, minimize waste, and prepare additional product as demand requires.

In [1239]:
#importing libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [1240]:
df = pd.read_csv("../data/processed/sales_merged.csv")
drink_df = df[df['Item'].isin(['Matcha Latte','Cold Brew','Lemonade'])]

Individual transactions were aggregated into total quantity sold for each beverage at each event. This transforms the data into one observation per drink per event, matching the forecasting objective of predicting total demand rather than individual purchases.

In [1241]:
model_df = (
    drink_df.groupby(
        ["Date", "Event_Type", "Location", "Item", "Avg_Temp","Weather_Condition"]
    )["Qty"]
    .sum()
    .reset_index()
)
model_df.head()

,Date,Event_Type,Location,Item,Avg_Temp,Weather_Condition,Qty
0,2026-05-23,Popup,Kill Devil Hills,Matcha Latte,77.0,Partly Cloudy,68.0
1,2026-05-30,Market,Manteo,Cold Brew,79.0,Sunny,51.0
2,2026-05-30,Market,Manteo,Lemonade,79.0,Sunny,21.0
3,2026-05-30,Market,Manteo,Matcha Latte,79.0,Sunny,87.0
4,2026-06-06,Market,Manteo,Cold Brew,86.0,Sunny,46.0


In [1242]:
model_df['Date']=pd.to_datetime(model_df['Date'])
model_df['Month']=model_df['Date'].dt.month
model_df['Weekend']= model_df['Date'].dt.dayofweek >=5

### Feature Engineering

To improve predictive performance, the following features were included:

- Month captures seasonal demand.
- Weekend indicates increased customer traffic.
- Event type and location account for differences between markets.
- Weather variables capture temperature-driven purchasing behavior.
- Beverage type allows a single model to forecast demand for multiple drinks.

In [1243]:
X = model_df[[
    "Event_Type",
    "Location",
    "Avg_Temp",
    "Weather_Condition",
    "Month",
    "Weekend",
    "Item"
]]
y=model_df['Qty']

### Train/Test Split

In [1244]:
X=pd.get_dummies(X,drop_first=True)

In [1245]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

### Baseline Model

In [1246]:
baseline_pred = np.full_like(y_test, y_train.mean())

In [1247]:
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
mae_baseline = mean_absolute_error(y_test, baseline_pred)
mse_baseline = mean_squared_error(y_test, baseline_pred)
rmse_baseline = np.sqrt(mse_baseline)

### Linear Regression

In [1248]:
lr = LinearRegression()
lr.fit(X_train, y_train)
pred_lr = lr.predict(X_test)

In [1249]:
mae_lr = mean_absolute_error(y_test, pred_lr)
rmse_lr = root_mean_squared_error(y_test, pred_lr)

### Random Forest

In [1250]:
rf=RandomForestRegressor(random_state=42)
rf.fit(X_train,y_train)
pred_rf=rf.predict(X_test)

In [1251]:
mae_rf = mean_absolute_error(y_test, pred_rf)
rmse_rf = root_mean_squared_error(y_test, pred_rf)

### Model Evaluation

In [1252]:
results = pd.DataFrame({
    "Model": ["Baseline", "Linear Regression", "Random Forest"],
    "MAE": [mae_baseline, mae_lr, mae_rf],
    "RMSE": [rmse_baseline, rmse_lr, rmse_rf]
})
results

,Model,MAE,RMSE
0,Baseline,13.743182,17.469420
1,Linear Regression,12.016931,14.204873
2,Random Forest,13.082727,16.321001


### Inventory Calculations

Predicted drink quantities are converted into gallons using standardized recipe measurements. Because both 12 oz and 16 oz drinks are sold, an average grams-per-drink value is calculated using the observed sales mix (70% 16 oz, 30% 12 oz).

In [1253]:
grams_12oz = {
    "Lemonade": 200,
    "Cold Brew": 100,
    "Matcha Latte": 40
}

grams_16oz = {
    "Lemonade": 300,
    "Cold Brew": 180,
    "Matcha Latte": 50
}

In [1254]:
def compute_effective_grams():

    mix_16 = 0.7
    mix_12 = 0.3

    effective_grams = {}

    for item in grams_12oz.keys():
        effective_grams[item] = (
            mix_16 * grams_16oz[item] +
            mix_12 * grams_12oz[item]
        )

    return effective_grams

In [1255]:
def inventory_calculator(pred_df):

    effective_grams = compute_effective_grams()

    GRAMS_PER_GALLON = 3300

    pred_df["Grams"] = pred_df.apply(
        lambda row: row["Predicted_Qty"] * effective_grams[row["Item"]],
        axis=1
    )

    pred_df["Gallons"] = pred_df["Grams"] / GRAMS_PER_GALLON

    return pred_df

In [1256]:
event_input_raw = pd.DataFrame({
    "Event_Type": ["Market"],
    "Location": ["Manteo"],
    "Avg_Temp": [85],
    "Weather_Condition": ["Sunny"],
    "Month": [6],
    "Weekend": [True],
})

In [1257]:
def build_event_grid(event_input_raw):

    items = ["Matcha Latte", "Cold Brew", "Lemonade"]

    rows = []

    for item in items:
        row = event_input_raw.copy()
        row["Item"] = item
        rows.append(row)

    return pd.concat(rows, ignore_index=True)

In [1258]:
event_grid = build_event_grid(event_input_raw)

event_encoded = pd.get_dummies(event_grid)
event_encoded = event_encoded.reindex(columns=X.columns, fill_value=0)

event_grid["Predicted_Qty"] = rf.predict(event_encoded)

In [1259]:
result = inventory_calculator(event_grid)

In [1260]:
result

,Event_Type,Location,Avg_Temp,Weather_Condition,Month,Weekend,Item,Predicted_Qty,Grams,Gallons
0,Market,Manteo,85,Sunny,6,True,Matcha Latte,68.73,3230.31,0.978882
1,Market,Manteo,85,Sunny,6,True,Cold Brew,58.08,9060.48,2.745600
2,Market,Manteo,85,Sunny,6,True,Lemonade,37.00,9990.00,3.027273
